# 17장 · 도구 왕복을 반복하는 기본 에이전트 루프

‘주말 운영 시간과 대출 기간을 함께 알려 주세요.’라고 물었습니다. 이 프로그램에서는 두 정보를 따로 조회합니다. 실행 기록을 이어 쓰고, 남은 호출 기회가 없으면 새로운 도구 요청을 실행하기 전에 멈추는 루프를 구성합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 첫 조회와 둘째 조회의 결과 구별하기

두 도구를 쓰기 위해 이 장에서는 이름을 get_hours와 get_loan_rule로 정했습니다. 함수 이름이 달라도 각 요청과 결과를 같은 call_id로 연결한다는 원칙은 16장과 같습니다.



**볼 곳**　‘이용 문의’에서 갈라지는 두 경우를 비교하세요. 위쪽은 ‘운영 시간’, 아래쪽은 ‘대출 기간’입니다.

**애니메이션의 의미**　첫 조회의 시간과 다음 조회의 대출 기간을 따로 모읍니다.


### 셀 01 · 첫 번째 도구 실행

제공된 execute는 이름·인자를 검사하고 조회 값을 JSON으로 묶습니다. 하나의 도구 결과부터 읽으세요.

In [1]:
import json

def execute(item):
    try:
        args=json.loads(item["arguments"])
        if not isinstance(args,dict): raise ValueError("객체 필요")
        if item["name"]=="get_hours" and set(args)=={"day"} and args["day"] in {"weekday","weekend"}:
            result={"ok":True,"hours":"09:00~18:00" if args["day"]=="weekday" else "10:00~16:00"}
        elif item["name"]=="get_loan_rule" and args=={}:
            result={"ok":True,"loan_days":14}
        else: raise ValueError("허용하지 않은 요청")
    except (ValueError,TypeError,KeyError):
        result={"ok":False,"error":"invalid_tool_request"}
    return {"type":"function_call_output","call_id":item["call_id"],"output":json.dumps(result,ensure_ascii=False)}

first = {"type": "function_call", "name": "get_hours", "call_id": "c1", "arguments": '{"day":"weekend"}'}
first_result = execute(first)
print(json.loads(first_result["output"])["hours"])

10:00~16:00


c1 요청에서 찾은 10:00~16:00입니다. 다음 요청은 대출 기간을 묻기 때문에 새 식별자 c2를 써야 합니다.

### 셀 02 · 두 번째 요청을 같은 기록에 붙이기

먼저 처리한 c1의 요청·응답 뒤에 c2의 요청과 결과를 넣습니다.

In [2]:
history = [{"role": "user", "content": "주말 시간과 대출 기간은?"}, first, first_result]
second = {"type": "function_call", "name": "get_loan_rule", "call_id": "c2", "arguments": "{}"}
history.append(second)
history.append(execute(second))
print(json.loads(history[-1]["output"])["loan_days"])

14


마지막 도구에서 반환한 값은 14일입니다. 아직 최종 안내는 기록에 없습니다. 모델은 두 결과를 본 뒤 답을 쓰기 위해 한 번 더 호출돼야 합니다.

## 종료를 판단하는 한 바퀴 읽기

모델 호출과 도구 실행은 따로 셉니다. 이번 대본의 첫 두 응답은 도구 호출이고 세 번째 응답은 답변입니다. 아래 함수는 진짜 모델 대신 준비된 응답을 순서대로 읽는 작은 루프입니다. 잘못된 ID와 중복을 검사하는 전체 수업 예제는 같은 종료 규칙을 더 엄격하게 적용합니다.



**볼 곳**　‘모델 판단’에서 시작해 ‘도구 실행’를 거친 뒤 ‘결과 다시 읽기’에서 다시 판단하는 흐름을 따라가세요.

**애니메이션의 의미**　새 도구 요청이 없으면 최종 답으로 루프를 끝냅니다.


### 셀 03 · 세 번째 응답에서 루프 끝내기

회차마다 먼저 응답 종류를 읽습니다. 최종 응답을 받거나 상한에 닿으면 함수를 빠져나갑니다.

모델이 더 요청할 도구가 없다고 답하면 이 루프는 끝납니다. 요청 ID 중복을 막으려면 execute(item)를 부르기 전에 이미 처리한 ID인지 확인해야 합니다. 실행 뒤에 ID를 비교하면 같은 도구가 이미 두 번 작동했을 수 있습니다.

In [3]:
def run_script(items, max_calls):
    history = [{"role": "user", "content": "주말 시간과 대출 기간은?"}]
    for number, item in enumerate(items, start=1):
        if number > max_calls:
            break
        history.append(item)
        if item["type"] == "message":
            return {"status": "completed", "history": history, "model_calls": number}
        if number == max_calls:
            return {"status": "model_limit", "history": history, "model_calls": number}
        history.append(execute(item))
    return {"status": "incomplete", "history": history, "model_calls": min(len(items), max_calls)}

answer = {"type": "message", "text": "주말 운영 시간은 10:00~16:00이고 대출 기간은 14일입니다."}
script = [first, second, answer]
trace = run_script(script, max_calls=3)
print(trace["model_calls"], trace["history"][-1]["text"])

3 주말 운영 시간은 10:00~16:00이고 대출 기간은 14일입니다.


세 번째 응답의 type이 message여서 최종 문장을 읽고 끝납니다. 대본에서 사용한 모의 모델 응답은 세 번, 도구 실행은 두 번입니다. 전체 구현은 미완료 응답도 따로 기록합니다.

### 과제 04 · 한도에 닿은 요청 조사하기

최대 두 번까지만 모델을 부를 수 있게 바꿔 종료 이유와 history 마지막 항목의 ID를 출력하세요.

In [4]:
# max_calls를 줄여 trace를 다시 얻으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>limited = run_script(script, max_calls=2)
print(limited[&quot;status&quot;], limited[&quot;history&quot;][-1][&quot;call_id&quot;])</code></pre>
<p>실행 결과</p><pre><code>model_limit c2</code></pre>
<p>model_limit c2입니다. c2 요청은 받았지만 실행 뒤 답을 받을 기회가 없으므로 둘째 도구 실행 전에 멈췄습니다. c2의 결과가 이미 있다고 해석하면 안 됩니다.</p>
</details>

## 실행했던 호출 ID가 다시 보인 경우

같은 요청이 두 번 들어왔다고 해서 같은 작업을 다시 실행해도 된다는 뜻은 아닙니다. 실행한 호출의 ID를 모아서 새 요청을 허용할지 먼저 판단합니다.



**볼 곳**　‘호출 ID’가 가운데 ‘실행 이력’에 도착한 뒤 오른쪽에 어떤 결과가 남는지 보세요.

**애니메이션의 의미**　실행 이력에 있는 ID를 다시 실행하지 않습니다.


### 과제 05 · 중복 식별자 찾기

이미 실행한 c1과 같은 ID가 다시 왔습니다. 실행 전 중복 여부를 표시하세요.

In [5]:
seen = {"c1"}
# first가 이미 실행된 요청인지 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>seen = {&quot;c1&quot;}
print(first[&quot;call_id&quot;] in seen)</code></pre>
<p>실행 결과</p><pre><code>True</code></pre>
<p>True이므로 같은 ID를 다시 실행하지 않습니다. 종료 상태를 남기면 요청 자체가 없던 경우와 구분해 원인을 찾을 수 있습니다.</p>
</details>

## 돌아보기

대화 기록에서 c2가 마지막일 때와 c2의 결과가 마지막일 때를 비교해 보세요. 같은 ID가 보이더라도 하나는 요청, 하나는 실행 뒤 기록입니다. 호출 상한을 두는 위치에 따라 실제 실행 횟수가 달라집니다.

### 두 도구와 최종 답변



**볼 곳**　시간 조회와 대출 조회를 각각 마친 뒤 누가 다음 행동을 고르는지 보세요.

**애니메이션의 의미**　에이전트는 도구 결과를 모델에 돌려주고, 추가 요청이 없을 때 최종 답을 반환합니다.
